# Bölüm 1: Markov Karar Süreçleri (MDP)
Bu notebook, Pekiştirmeli Öğrenme'nin temel matematiksel altyapısını oluşturan **Markov Karar Süreçleri (MDP)** kavramlarını uygulamalı olarak ele almaktadır.

---
**Konular:**
1. MDP Tanımı ve Bileşenleri
2. Politika ve Değer Fonksiyonları
3. Bellman Denklemleri
4. Değer İterasyonu (Value Iteration)
5. Politika İterasyonu (Policy Iteration)
6. Uygulama: Envanter Kontrolü ve Kumar Problemi


## 1.1 Kütüphaneler ve Kurulum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Grafik ayarları
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Kütüphaneler başarıyla yüklendi!")
print(f"NumPy versiyonu: {np.__version__}")

## 1.2 MDP Nedir?

Bir **Markov Karar Süreci** şu unsurlardan oluşur:

$$\mathcal{M} = (\mathcal{X}, \mathcal{A}, P_0)$$

| Sembol | Anlam |
|--------|-------|
| $\mathcal{X}$ | Durum uzayı (state space) |
| $\mathcal{A}$ | Eylem uzayı (action space) |
| $P_0(\cdot \mid x, a)$ | Geçiş çekirdegi (transition kernel) |
| $r(x, a)$ | Anlık ödül fonksiyonu |
| $\gamma \in [0,1)$ | İndirim faktörü |

**Dinamik:** Her adımda ajan durumu $X_t$ gözlemler, $A_t$ eylemini seçer ve sistem güncellenir:
$$(X_{t+1}, R_{t+1}) \sim P_0(\cdot \mid X_t, A_t)$$

**Hedef:** Toplam indirimlenmiş ödülü maksimize etmek:
$$R = \sum_{t=0}^{\infty} \gamma^t R_{t+1}$$


## 1.3 MDP Sınıfı Implementasyonu

In [ ]:
class MDP:
    """
    Sonlu Markov Karar Süreci (Finite MDP)
    
    Parametreler:
    -----------
    states     : Durum listesi
    actions    : Eylem listesi
    transitions: {(s, a): {s': prob}} sözlüğü
    rewards    : {(s, a): float} ödül fonksiyonu
    gamma      : İndirim faktörü
    """
    def __init__(self, states, actions, transitions, rewards, gamma=0.95):
        self.states = states
        self.actions = actions
        self.transitions = transitions
        self.rewards = rewards
        self.gamma = gamma
        self.n_states = len(states)
        self.n_actions = len(actions)
        self.state_idx = {s: i for i, s in enumerate(states)}
        self.action_idx = {a: i for i, a in enumerate(actions)}
    
    def step(self, state, action):
        """Bir adım simüle eder: (yeni_durum, ödül) döner"""
        next_states = list(self.transitions[(state, action)].keys())
        probs = list(self.transitions[(state, action)].values())
        next_state = np.random.choice(next_states, p=probs)
        reward = self.rewards.get((state, action), 0.0)
        return next_state, reward
    
    def bellman_policy(self, V, policy):
        """Politika için Bellman güncellemesi"""
        V_new = np.zeros(self.n_states)
        for i, s in enumerate(self.states):
            a = policy[s]
            V_new[i] = self.rewards.get((s, a), 0)
            for s_next, prob in self.transitions.get((s, a), {}).items():
                j = self.state_idx[s_next]
                V_new[i] += self.gamma * prob * V[j]
        return V_new
    
    def bellman_optimality(self, V):
        """Bellman optimallik operatörü T*"""
        V_new = np.zeros(self.n_states)
        for i, s in enumerate(self.states):
            q_vals = []
            for a in self.actions:
                if (s, a) in self.transitions:
                    q = self.rewards.get((s, a), 0)
                    for s_next, prob in self.transitions[(s, a)].items():
                        j = self.state_idx[s_next]
                        q += self.gamma * prob * V[j]
                    q_vals.append(q)
            V_new[i] = max(q_vals) if q_vals else 0
        return V_new

print("MDP sınıfı tanımlandı ✓")

## 1.4 Örnek: Izgara Dünyası (GridWorld)

Klasik bir pekiştirmeli öğrenme ortamı olan 4x4'lük ızgara dünyasını kullanalım.

```
 0  1  2  3
 4  5  6  7
 8  9 10 11
12 13 14 15(G)
```
- **Hedef:** 15. hücreye ulaşmak (+1 ödül)
- **Tuzak:** 11. hücre (-1 ödül, biter)
- **Hareketler:** Yukarı, Aşağı, Sol, Sağ
- **Kayma olasılığı:** %20 (duvarlar sinyali yansıtır)


In [ ]:
def create_gridworld(grid_size=4, gamma=0.95, slip_prob=0.1):
    """4x4 Izgara Dünyası MDP'si oluşturur"""
    n = grid_size
    states = list(range(n * n))
    actions = ['up', 'down', 'left', 'right']
    goal_state = n * n - 1    # Sağ alt köşe
    trap_state = n * n - 5    # 11. hücre (4x4 için)
    
    action_delta = {'up': -n, 'down': n, 'left': -1, 'right': 1}
    
    def is_valid(s):
        return 0 <= s < n * n
    
    def apply_action(s, a):
        row, col = s // n, s % n
        if a == 'up' and row > 0: return s - n
        if a == 'down' and row < n-1: return s + n
        if a == 'left' and col > 0: return s - 1
        if a == 'right' and col < n-1: return s + 1
        return s  # Duvara çarpma: yerinde kal
    
    transitions = {}
    rewards = {}
    
    for s in states:
        if s in [goal_state, trap_state]:  # Terminal durumlar
            for a in actions:
                transitions[(s, a)] = {s: 1.0}
                rewards[(s, a)] = 0.0
            continue
        
        for a in actions:
            # Ana hareket
            main_next = apply_action(s, a)
            # Kayma: diğer yönler
            other_actions = [x for x in actions if x != a]
            trans = {main_next: 1 - slip_prob}
            
            for oa in other_actions:
                slip_next = apply_action(s, oa)
                trans[slip_next] = trans.get(slip_next, 0) + slip_prob / len(other_actions)
            
            transitions[(s, a)] = trans
            
            # Ödül
            if main_next == goal_state:
                rewards[(s, a)] = 1.0
            elif main_next == trap_state:
                rewards[(s, a)] = -1.0
            else:
                rewards[(s, a)] = -0.01  # Küçük adım cezası
    
    mdp = MDP(states, actions, transitions, rewards, gamma)
    mdp.goal_state = goal_state
    mdp.trap_state = trap_state
    mdp.grid_size = n
    return mdp

gridworld = create_gridworld(gamma=0.95)
print(f"GridWorld oluşturuldu:")
print(f"  Durum sayısı: {gridworld.n_states}")
print(f"  Eylem sayısı: {gridworld.n_actions}")
print(f"  İndirim faktörü: {gridworld.gamma}")
print(f"  Hedef durum: {gridworld.goal_state}")
print(f"  Tuzak durum: {gridworld.trap_state}")

## 1.5 Değer Fonksiyonları ve Bellman Denklemleri

### Politika Değer Fonksiyonu
Bir $\pi$ politikası altındaki değer fonksiyonu:
$$V^\pi(x) = \mathbb{E}\left[\sum_{t=0}^{\infty} \gamma^t R_{t+1} \mid X_0 = x\right]$$

**Bellman Denklemi** (politika $\pi$ için):
$$V^\pi(x) = r(x, \pi(x)) + \gamma \sum_{y \in \mathcal{X}} P(x, \pi(x), y) V^\pi(y)$$

### Optimal Değer Fonksiyonu
$$V^*(x) = \sup_{\pi} V^\pi(x)$$

**Bellman Optimallik Denklemi:**
$$V^*(x) = \sup_{a \in \mathcal{A}} \left[ r(x,a) + \gamma \sum_{y \in \mathcal{X}} P(x,a,y) V^*(y) \right]$$


## 1.6 Değer İterasyonu (Value Iteration)

In [ ]:
def value_iteration(mdp, tol=1e-6, max_iter=1000, verbose=True):
    """
    Değer İterasyonu Algoritması
    
    V_{k+1} = T* V_k
    
    T* Bellman optimallik operatörüdür.
    Banach sabit nokta teoreminden, V_k → V* geometrik hızda.
    """
    V = np.zeros(mdp.n_states)
    history = [V.copy()]
    errors = []
    
    for iteration in range(max_iter):
        V_new = mdp.bellman_optimality(V)
        error = np.max(np.abs(V_new - V))
        errors.append(error)
        V = V_new.copy()
        history.append(V.copy())
        
        if error < tol:
            if verbose:
                print(f"Yakınsama sağlandı! İterasyon: {iteration+1}, Hata: {error:.2e}")
            break
    
    # Greedy politika çıkar
    policy = {}
    Q = np.zeros((mdp.n_states, mdp.n_actions))
    for i, s in enumerate(mdp.states):
        for j, a in enumerate(mdp.actions):
            if (s, a) in mdp.transitions:
                Q[i, j] = mdp.rewards.get((s, a), 0)
                for s_next, prob in mdp.transitions[(s, a)].items():
                    k = mdp.state_idx[s_next]
                    Q[i, j] += mdp.gamma * prob * V[k]
        best_action_idx = np.argmax(Q[i])
        policy[s] = mdp.actions[best_action_idx]
    
    return V, policy, errors, history

# Değer iterasyonunu çalıştır
V_star, optimal_policy, errors, history = value_iteration(gridworld)

# Sonuçları görselleştir
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) Optimal değer fonksiyonu
n = gridworld.grid_size
V_grid = V_star.reshape(n, n)
im = axes[0].imshow(V_grid, cmap='RdYlGn', aspect='auto')
axes[0].set_title('Optimal Değer Fonksiyonu V*(s)', fontweight='bold')
for i in range(n):
    for j in range(n):
        s = i * n + j
        axes[0].text(j, i, f'{V_grid[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=axes[0])

# 2) Optimal politika
arrow_map = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}
policy_grid = np.zeros((n, n))
axes[1].set_xlim(-0.5, n-0.5)
axes[1].set_ylim(-0.5, n-0.5)
axes[1].set_title('Optimal Politika π*(s)', fontweight='bold')
for i in range(n):
    for j in range(n):
        s = i * n + j
        if s == gridworld.goal_state:
            axes[1].text(j, n-1-i, 'G✓', ha='center', va='center', 
                        fontsize=14, color='green', fontweight='bold')
        elif s == gridworld.trap_state:
            axes[1].text(j, n-1-i, 'T✗', ha='center', va='center', 
                        fontsize=14, color='red', fontweight='bold')
        else:
            axes[1].text(j, n-1-i, arrow_map[optimal_policy[s]], 
                        ha='center', va='center', fontsize=18)
axes[1].set_xticks(range(n))
axes[1].set_yticks(range(n))
axes[1].grid(True, alpha=0.5)

# 3) Yakınsama eğrisi
axes[2].semilogy(errors, 'b-o', markersize=4)
axes[2].set_xlabel('İterasyon')
axes[2].set_ylabel('Maksimum Hata (log ölçek)')
axes[2].set_title('Değer İterasyonu Yakınsama', fontweight='bold')
axes[2].axhline(y=1e-6, color='r', linestyle='--', label='Tolerans (1e-6)')
axes[2].legend()

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_value_iteration.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nOptimal politika (bazı durumlar):")
for s in [0, 1, 4, 5, 9, 10]:
    print(f"  Durum {s:2d}: {optimal_policy[s]}")

## 1.7 Politika İterasyonu (Policy Iteration)

In [ ]:
def policy_evaluation(mdp, policy, tol=1e-8, max_iter=500):
    """
    Politika Değerlendirme: Bellman doğrusal sistemini çözer
    V^pi = r_pi + gamma * P_pi * V^pi
    """
    V = np.zeros(mdp.n_states)
    for _ in range(max_iter):
        V_new = mdp.bellman_policy(V, policy)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return V

def policy_improvement(mdp, V):
    """
    Politika İyileştirme: Q* bilgisini kullanarak greedy politika türet
    pi_new(s) = argmax_a Q(s,a)
    """
    policy = {}
    for i, s in enumerate(mdp.states):
        best_val = -np.inf
        best_action = mdp.actions[0]
        for a in mdp.actions:
            if (s, a) not in mdp.transitions:
                continue
            q = mdp.rewards.get((s, a), 0)
            for s_next, prob in mdp.transitions[(s, a)].items():
                j = mdp.state_idx[s_next]
                q += mdp.gamma * prob * V[j]
            if q > best_val:
                best_val = q
                best_action = a
        policy[s] = best_action
    return policy

def policy_iteration(mdp, verbose=True):
    """
    Politika İterasyonu Algoritması:
    1. Politikayı değerlendir: V^pi_k
    2. Politikayı iyileştir: pi_{k+1} = greedy(V^pi_k)
    3. Değişim olmayana dek tekrarla
    """
    # Rastgele başlangıç politikası
    policy = {s: np.random.choice(mdp.actions) for s in mdp.states}
    n_iters = 0
    policy_changes = []
    
    while True:
        # Adım 1: Politika değerlendirme
        V = policy_evaluation(mdp, policy)
        
        # Adım 2: Politika iyileştirme
        new_policy = policy_improvement(mdp, V)
        
        # Kaç durum değişti?
        changes = sum(1 for s in mdp.states if policy.get(s) != new_policy[s])
        policy_changes.append(changes)
        n_iters += 1
        
        if verbose:
            print(f"  İterasyon {n_iters}: {changes} durum değişti")
        
        if changes == 0:
            if verbose:
                print(f"Politika yakınsadı! Toplam iterasyon: {n_iters}")
            break
        
        policy = new_policy
    
    return V, policy, policy_changes

print("Politika İterasyonu çalışıyor...")
V_pi, optimal_policy_pi, policy_changes = policy_iteration(gridworld)

# Karşılaştırma
print(f"\nDeğer İterasyonu V* ile Politika İterasyonu V* arasındaki max fark:")
print(f"  max|V_vi - V_pi| = {np.max(np.abs(V_star - V_pi)):.2e}")
print("\nPolitika İterasyonu daha az iterasyonla aynı sonuca ulaşır!")

# Politika değişim grafiği
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(policy_changes)+1), policy_changes, color='steelblue', alpha=0.8)
plt.xlabel('Politika İterasyonu Adımı')
plt.ylabel('Değişen Durum Sayısı')
plt.title('Politika İterasyonu: Her Adımdaki Değişim Sayısı')
plt.xticks(range(1, len(policy_changes)+1))
plt.savefig('/mnt/user-data/outputs/ch1_policy_iteration.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.8 Uygulama: Envanter Kontrolü

Kitabın 1. Bölümündeki **Örnek 1.1**'i implemente edelim.

Günlük envanter yönetimi problemi:
- $X_t$: Akşam envanter büyüklüğü
- $A_t$: Sipariş miktarı
- $D_{t+1} \sim \text{Poisson}(\lambda)$: Stokastik talep

$$X_{t+1} = ((X_t + A_t) \wedge M - D_{t+1})^+$$

**Ödül:**
$$R_{t+1} = p \cdot \min(X_t + A_t, D_{t+1}) - c \cdot A_t - K \cdot \mathbf{1}[A_t > 0] - h \cdot X_{t+1}$$


In [ ]:
from scipy.stats import poisson

def create_inventory_mdp(M=10, lam=3, p=10, c=2, K=5, h=1, gamma=0.95):
    """
    Envanter Kontrolü MDP'si (Örnek 1.1 - Szepesvári)
    
    M   : Maksimum envanter kapasitesi
    lam : Poisson talep ortalaması
    p   : Satış fiyatı
    c   : Birim sipariş maliyeti
    K   : Sabit sipariş maliyeti
    h   : Envanter tutma maliyeti (birim başına)
    """
    states = list(range(M + 1))        # 0, 1, ..., M
    actions = list(range(M + 1))       # Sipariş miktarı: 0, 1, ..., M
    
    # Talep olasılıkları (Poisson)
    max_demand = 3 * M
    demand_probs = [poisson.pmf(d, lam) for d in range(max_demand + 1)]
    # Normalize et
    total = sum(demand_probs)
    demand_probs = [p_ / total for p_ in demand_probs]
    
    transitions = {}
    rewards = {}
    
    for x in states:
        for a in actions:
            # Geçerlilik: stok + sipariş <= M
            if x + a > M:
                continue
            
            trans = {}
            total_reward = 0.0
            
            for d, dp in enumerate(demand_probs):
                if dp < 1e-10:
                    continue
                x_after_order = min(x + a, M)
                x_next = max(x_after_order - d, 0)
                sold = min(x_after_order, d)
                
                # Ödül hesabı
                order_cost = (K + c * a) if a > 0 else 0
                holding_cost = h * x_next
                revenue = p * sold
                r = revenue - order_cost - holding_cost
                
                trans[x_next] = trans.get(x_next, 0) + dp
                total_reward += dp * r
            
            transitions[(x, a)] = trans
            rewards[(x, a)] = total_reward
    
    return MDP(states, actions, transitions, rewards, gamma)

# MDP oluştur
inv_mdp = create_inventory_mdp(M=15, lam=4)
print(f"Envanter MDP oluşturuldu: {inv_mdp.n_states} durum, {inv_mdp.n_actions} eylem")

# Çöz
print("\nDeğer iterasyonu çözüyor...")
V_inv, policy_inv, errors_inv, _ = value_iteration(inv_mdp, verbose=False)

# Optimal politikayı görselleştir
states = inv_mdp.states
opt_orders = [int(policy_inv[x]) for x in states]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Optimal sipariş politikası
axes[0].bar(states, opt_orders, color='darkorange', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Mevcut Envanter Seviyesi (x)', fontsize=12)
axes[0].set_ylabel('Optimal Sipariş Miktarı A*(x)', fontsize=12)
axes[0].set_title('Optimal Envanter Politikası\n(s,S) Tipi Politika', fontweight='bold')
axes[0].axhline(y=0, color='black', linewidth=0.5)

# (s,S) politikasını bul
# Sipariş eşiği (s): ne zaman sipariş verilir?
order_threshold = max([x for x in states if policy_inv[x] > 0], default=0)
axes[0].axvline(x=order_threshold, color='red', linestyle='--', 
                label=f's (sipariş eşiği) = {order_threshold}')
axes[0].legend()

# Sağ: Optimal değer fonksiyonu
axes[1].plot(states, V_inv, 'b-o', markersize=6, linewidth=2)
axes[1].set_xlabel('Envanter Seviyesi (x)', fontsize=12)
axes[1].set_ylabel('V*(x)', fontsize=12)
axes[1].set_title('Optimal Değer Fonksiyonu\n(Beklenen İndirimlenmiş Toplam Kar)', fontweight='bold')
axes[1].fill_between(states, V_inv, alpha=0.2)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_inventory.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSipariş eşiği (s): Envanter <= {order_threshold} ise sipariş ver")

## 1.9 Kumar Problemi (Gambler's Problem — Örnek 1.2)

Kitabın Örnek 1.2'si: Kumarbazın serveti $w^*$'a ulaşana kadar devam eder.

$$X_{t+1} = (1 + S_{t+1} A_t) X_t, \ S_{t+1} \in \{-1, +1\}$$

Hedef: $P(X_t \rightarrow w^*)$ olasılığını maksimize etmek.


In [ ]:
def create_gambler_mdp(w_star=100, p_win=0.4, gamma=1.0):
    """
    Kumar Problemi (Örnek 1.2 - Szepesvári)
    p_win: kazanma olasılığı (< 0.5: dezavantajlı oyun)
    """
    states = list(range(w_star + 1))   # 0, 1, ..., w*
    
    transitions = {}
    rewards = {}
    
    for x in states:
        if x == 0 or x == w_star:      # Terminal durumlar
            for a in range(x + 1):
                transitions[(x, a)] = {x: 1.0}
                rewards[(x, a)] = 0.0
            continue
        
        # Eylemler: 1, 2, ..., min(x, w*-x)
        max_stake = min(x, w_star - x)
        actions_x = list(range(1, max_stake + 1))
        
        for a in actions_x:
            win_state = min(x + a, w_star)
            lose_state = max(x - a, 0)
            
            transitions[(x, a)] = {win_state: p_win, lose_state: 1 - p_win}
            # Hedef duruma ulaşıldığında +1 ödül
            rewards[(x, a)] = p_win if win_state == w_star else 0.0
    
    # Tüm durumlar için geçerli eylem seti
    all_actions = list(range(1, w_star // 2 + 1))
    
    # Sadece geçerli durum-eylem çiftlerini kullan
    valid_pairs = list(transitions.keys())
    
    mdp = MDP(states, all_actions, transitions, rewards, gamma)
    return mdp, w_star

gambler_mdp, w_star = create_gambler_mdp(w_star=100, p_win=0.4)
print(f"Kumar MDP: {gambler_mdp.n_states} durum")

# Değer iterasyonuyla çöz (birkaç p değeri için)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p_win, color, label in [(0.4, 'red', 'p=0.4 (dezavantajlı)'),
                              (0.5, 'blue', 'p=0.5 (adil)'),
                              (0.6, 'green', 'p=0.6 (avantajlı)')]:
    g_mdp, _ = create_gambler_mdp(w_star=100, p_win=p_win)
    V_g, pol_g, _, _ = value_iteration(g_mdp, tol=1e-9, verbose=False)
    axes[0].plot(g_mdp.states, V_g, color=color, linewidth=2, label=label)

axes[0].set_xlabel('Servet (x)', fontsize=12)
axes[0].set_ylabel("V*(x) = P(w*'a ulaşma)", fontsize=12)
axes[0].set_title('Kumar Problemi: Optimal Kazanma Olasılığı', fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0, 100)

# p=0.4 için optimal bahis miktarı
g_mdp_04, _ = create_gambler_mdp(w_star=100, p_win=0.4)
V_g04, pol_g04, _, _ = value_iteration(g_mdp_04, tol=1e-9, verbose=False)
stakes = [pol_g04.get(x, 0) for x in g_mdp_04.states]

axes[1].bar(g_mdp_04.states[1:-1], stakes[1:-1], color='darkblue', alpha=0.7)
axes[1].set_xlabel('Servet (x)', fontsize=12)
axes[1].set_ylabel('Optimal Bahis Miktarı A*(x)', fontsize=12)
axes[1].set_title('Optimal Bahis Politikası (p=0.4)', fontweight='bold')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/ch1_gambler.png', dpi=150, bbox_inches='tight')
plt.show()
print("Kumar problemi çözüldü!")

## 1.10 Özet ve Değerlendirme

| Kavram | Açıklama |
|--------|----------|
| **MDP** | $(\mathcal{X}, \mathcal{A}, P_0)$ üçlüsü |
| **Değer Fonksiyonu** | $V^\pi(x) = \mathbb{E}[\sum \gamma^t R_{t+1} \mid X_0=x]$ |
| **Bellman Denklemi** | $V^\pi = T^\pi V^\pi$ (doğrusal denklem) |
| **Bellman Optimallik** | $V^* = T^* V^*$ (doğrusal olmayan) |
| **Değer İterasyonu** | $V_{k+1} = T^* V_k$, geometrik yakınsama |
| **Politika İterasyonu** | Değerlendirme + iyileştirme döngüsü |

### Teorik Garanti (Banach Sabit Nokta Teoremi)
$T^*$ operatörü $\gamma < 1$ için maksimum-norm büzülmesidir:
$$\|T^* V - T^* W\|_\infty \leq \gamma \|V - W\|_\infty$$

Bu yüzden değer iterasyonu her zaman $V^*$'a yakınsar!

### Sonraki Bölüm
Bölüm 2'de bu değer fonksiyonlarını **model bilinmeden** (model-free) nasıl tahmin edeceğimizi göreceğiz.
